In [44]:
# --- PART 1 -----
# a stream flows down from "S". at each gateway (^), the stream will split in two
# keep in mind that neighboring gateways may split into the same lane.

# password: total number of times the ^ are activated


# EX:

# .......S.......   .......S.......   .......S.......   .......S.......
# ...............   .......|.......   .......|.......   .......|.......
# .......^.......   .......^.......   ......|^|......   ......|^|......
# ...............   ...............   ......|.|......   ......|.|......
# ......^.^......   ......^.^......   ......^.^......   .....|^|^|.....
# ...............   ...............   ...............   .....|.|.|.....
# .....^.^.^.....   .....^.^.^.....   .....^.^.^.....   ....|^|^|^|....
# ...............   ...............   ...............   ....|.|.|.|....
# ....^.^...^....   ....^.^...^....   ....^.^...^....   ...|^|^|||^|...
# ...............   ...............   ...............   ...|.|.|||.|...
# ...^.^...^.^...   ...^.^...^.^...   ...^.^...^.^...   ..|^|^|||^|^|..
# ...............   ...............   ...............   ..|.|.|||.|.|..
# ..^...^.....^..   ..^...^.....^..   ..^...^.....^..   .|^|.|^||.||^|.
# ...............   ...............   ...............   .|.|.|.||.||.|.
# .^.^.^.^.^...^.   .^.^.^.^.^...^.   .^.^.^.^.^...^.   |^|^|^|^|^|||^|
# ...............   ...............   ...............   |.|.|.|.|.|||.|

# password = 21

In [45]:
with open(file="day7.txt", encoding="utf-8") as f:
    data = [list(line.rstrip()) for line in f]

starting_point = data[0].index("S")
active_currents = {starting_point}
activated_gateways = 0

for line in data:
    if "^" in line:
        gateway_indices = [i for i, val in enumerate(line) if val == "^"]

        for gateway in gateway_indices:
            if gateway in active_currents:
                activated_gateways += 1
                active_currents.remove(gateway)
                active_currents.update([gateway - 1, gateway + 1])

activated_gateways

1546

In [46]:
# --- PART 2 -----
# Now gateways only split in one direction, i.e. either left or right.

# password: the number of possible permutations the stream might take


# EXample permutations:
# .......S.......   .......S.......   .......S.......   .......S.......
# .......|.......   .......|.......   .......|.......   .......|.......
# ......|^.......   .......^|......   ......|^.......   ......|^.......
# ......|........   ........|......   ......|........   ......|........
# .....|^.^......   ......^|^......   ......^|^......   ......^|^......
# .....|.........   .......|.......   .......|.......   .......|.......
# ....|^.^.^.....   .....^.^|^.....   .....^|^.^.....   .....^|^.^.....
# ....|..........   ........|......   ......|........   ......|........
# ...|^.^...^....   ....^.^.|.^....   ....^|^...^....   ....^.^|..^....
# ...|...........   ........|......   .....|.........   .......|.......
# ..|^.^...^.^...   ...^.^..|^.^...   ...^.^|..^.^...   ...^.^.|.^.^...
# ..|............   ........|......   ......|........   .......|.......
# .|^...^.....^..   ..^...^.|...^..   ..^...^|....^..   ..^...^|....^..
# .|.............   ........|......   .......|.......   .......|.......
# |^.^.^.^.^...^.   .^.^.^.^|^...^.   .^.^.^.^|^...^.   .^.^.^.^|^...^.
# |..............   ........|......   ........|......   ........|......

# password = 40

In [ ]:
# FAILED ATTEMPT

# Binary node (i.e. left and right arm)
class Node:
    # incrementing ID
    _counter = 1000

    def __init__(self, position, left=None, right=None):
        self.position = position
        self.left = left
        self.right = right

        self.val = Node._counter
        Node._counter += 1

    def __str__(self):
        left_val = self.left.val if self.left else None
        right_val = self.right.val if self.right else None
        return f"{left_val} <- {self.val} -> {right_val}"


# read file and create tree-structure
with open(file="day7.txt", encoding="utf-8") as f:
    data = [list(line.rstrip()) for line in f]

starting_point = data[0].index("S")
active_currents = {starting_point}
tree = [Node(starting_point, left=starting_point)]

for line in data:
    if "^" in line:
        gateway_indices = [i for i, val in enumerate(line) if val == "^"]
        unassigned_nodes = [Node(position=pos) for pos in gateway_indices]

        for gateway, node in zip(gateway_indices, unassigned_nodes):

            if node.position in active_currents:
                node.left = node.position - 1
                node.right = node.position + 1

                for parent_node in tree:
                    if parent_node.left == node.position:
                        parent_node.left = node
                    if parent_node.right == node.position:
                        parent_node.right = node

                tree.append(node)

            if gateway in active_currents:
                active_currents.remove(gateway)
                active_currents.update([gateway - 1, gateway + 1])


# cleaning up gateways where left/right do not point to another node
for node in tree:
    try:
        node.left.val
    except AttributeError:
        node.left = None
    try:
        node.right.val
    except AttributeError:
        node.right = None

# removing head (S)
del tree[0]

In [ ]:
# SOLUTION FOUND ONLINE

import collections

def propagate_tachyons(tachyons, splitters):
    new_tachyons = collections.defaultdict(int)
    for tachyon_pos, tachyon_count in tachyons.items():
        if tachyon_pos in splitters:
            new_tachyons[tachyon_pos - 1] += tachyon_count
            new_tachyons[tachyon_pos + 1] += tachyon_count
        else:
            new_tachyons[tachyon_pos] += tachyon_count

    return new_tachyons


with open("day7.txt") as f:
    start_line = f.readline()
    start_point = start_line.find("S")
    tachyons = collections.defaultdict(int)
    tachyons[start_point] += 1
    splitters = []
    for line in f:
        splitter_row = set([idx for idx, char in enumerate(line) if char == "^"])
        if len(splitter_row) > 0:
            splitters.append(splitter_row)

out = 0
for row in splitters:
    tachyons = propagate_tachyons(tachyons, row)

print(sum(tachyons.values()))

13883459503480
